# Track Optimization with GIF Visualization

This notebook runs Adam optimization from a **randomly smeared initial guess** and generates an animated GIF showing the optimization progression.

**Approach:**
- Skip grid search stages (0-3)
- Generate initial parameters by smearing around true values
- Run Adam optimizer from this distant starting point
- Visualize convergence with animated GIF

**Smearing Parameters:**
- Position: Gaussian smearing (configurable sigma in meters)
- Direction: Angular smearing (configurable sigma in degrees)
- Energy: Relative smearing (configurable percentage)
- t0: Gaussian smearing (configurable sigma in ns)

## Cell 1: Environment Setup and Imports

In [ ]:
import sys
sys.path.append('..')

# Standard imports
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import plotly.graph_objects as go
from tqdm import tqdm
import json
import os
import time
import pickle
import glob
import uproot
import shutil
from pathlib import Path
from jax import jit, value_and_grad
import optax
import subprocess
from PIL import Image

# LUCiD imports
from lucid.geometry import generate_detector
from lucid.generate import read_photon_data_from_photonsim
from lucid.simulation import setup_event_simulator
from lucid.utils import load_range_params, check_track_endpoint_in_detector
from lucid.detector_params import ParticleParams, load_detector_params

# Optimization imports
from lucid.optimization.grid_search import (
    load_optimization_config, 
    get_detector_bounds, 
    hierarchical_position_grid_search
)
from lucid.optimization.utils.functions import (
    hierarchical_direction_search_cone, 
    energy_scan_optimization,
    cartesian_to_spherical, 
    spherical_to_cartesian, 
    performance_summary,
    estimate_muon_energy_from_photon_count
)

# New likelihood-based losses
from lucid.losses import (
    first_arrival_nll,
    poisson_nll,
    origin_time_loss_configurable,
    TAU_VTX_PARAM_A,
    TAU_VTX_PARAM_B,
    TAU_VTX_PARAM_C,
)

# Visualization imports
from lucid.optimization.utils.visualization import (
    create_event_3D_visualization, 
    create_optimization_path_3d_visualization
)
from lucid.optimization.utils.geometry import create_cylinder_surface, create_sphere_surface
from lucid.optimization.run import load_config
from lucid.visualization import create_detector_comparison_display

print('All imports successful!')

## Cell 2: Configuration Selection

| Config | nphot |
|--------|-------|
| 0 | 5,000 |
| 1 | 10,000 |
| 2 | 25,000 |
| 3 | 50,000 |
| 4 | 100,000 |
| **5** | **150,000** |
| 6 | 200,000 |
| 7 | 300,000 |
| 8 | 500,000 |

In [ ]:
# CONFIG PARAMETERIZATION - Change this to switch configs (0-8)
CONFIG_INDEX = 5

config_dir = Path('../s3df_jobs/nrays_config')
config_path = config_dir / f'opt_config_{CONFIG_INDEX}.json'
script_path = config_dir / 'create_configs.py'

if not config_path.exists():
    print('Config file not found. Creating it...')
    subprocess.run([sys.executable, script_path.name], cwd=config_dir, check=True)
    print('Config file successfully created.')
else:
    print('Config file already exists.')

print(config_path)
adam_config = load_config(config_path)
config = load_optimization_config(config_path)

# Add default values
if 'optimization_params' not in config:
    config['optimization_params'] = {}
config['optimization_params'].setdefault('damping_factor', 0.998)

if 'adam_optimizer' not in config:
    config['adam_optimizer'] = {}
config['adam_optimizer'].setdefault('learning_rate', 0.2)
config['adam_optimizer'].setdefault('b1', 0.9)
config['adam_optimizer'].setdefault('b2', 0.999)
config['adam_optimizer'].setdefault('eps', 1e-8)

print(f'Loaded Config {CONFIG_INDEX}')
print('=' * 50)
print(f"nphot: {config['basic_config']['nphot']:,}")
print(f"temperature: {config['basic_config']['temperature']}")
print(f"detector: {config['basic_config']['default_json_filename']}")

## Cell 3: Detector and Simulator Setup

In [ ]:
# Physics config for detector parameters
PHYSICS_CONFIG = '../config/SK_physics_config.json'

# Extract basic configuration
default_json_filename = '../config/SK_geom_config.json'
data_dir = '../data/water/muon/'
TEMPERATURE = 0.10
N_EVENTS = 1  # Process 1 event for GIF generation
K = config['basic_config']['k']
Nphot = 50_000
C_MEDIUM = config['basic_config']['c_medium']
qe = config['detector_params']['qe']

# Setup detector
print(f'Setting up detector from: {default_json_filename}')
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Get detector bounds
detector_bounds = get_detector_bounds(detector)
DETECTOR_R = detector_bounds.get('r', None)
DETECTOR_H = detector_bounds.get('H', None)

print(f"Detector type: {detector_bounds['type']}")
print(f'Detector R: {DETECTOR_R:.2f} m')
print(f'Detector H: {DETECTOR_H:.2f} m')
print(f'Number of sensors: {NUM_DETECTORS}')

# Load range parametrization
range_params = load_range_params('muon', 'water')
print(f"Loaded range parametrization: {range_params['description']}")

# Setup simulators
print('Setting up simulators...')
prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, 
    max_sensors_per_cell=4, K=K, is_data=False, hit_mode='per_photon',
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

data_simulator = setup_event_simulator(
    default_json_filename, Nphot, temperature=0.0, 
    K=K, is_data=True, is_calibration=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

detector_params = load_detector_params(PHYSICS_CONFIG)
print('Simulators ready!')
print(f'Detector params: scatter_length={detector_params.scatter_length}, wall_reflection_rate={detector_params.wall_reflection_rate}')

## Cell 4: Event Generation Function

In [ ]:
def generate_event_data(event_idx, random_key, data_dir, data_simulator,
                       detector_bounds, fraction=0.9):
    '''Generate a single event with random parameters within detector bounds.'''
    root_files = sorted(glob.glob(os.path.join(data_dir, '*.root')))
    if not root_files:
        raise ValueError(f'No .root files found in directory: {data_dir}')

    file_select_key, random_key = jax.random.split(random_key)
    file_idx = jax.random.randint(file_select_key, shape=(), minval=0, maxval=len(root_files))
    data_file = root_files[int(file_idx)]

    with uproot.open(data_file) as file:
        tree = file['OpticalPhotons']
        n_entries = tree.num_entries

    entry_idx = event_idx % n_entries
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

    photon_origins = photon_data['photon_origins']
    photon_directions = photon_data['photon_directions']
    photon_times = photon_data['photon_times']
    N = len(photon_origins)
    
    padding_size = max(0, 1_000_000 - N)
    photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                        mode='constant', constant_values=0)

    default_direction = jnp.array([0.0, 0.0, 1.0])
    padding_directions = jnp.tile(default_direction, (padding_size, 1))
    if padding_size > 0:
        photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
    else:
        photon_data['photon_directions'] = photon_directions

    photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size), mode='constant', constant_values=0)
    photon_data['N'] = N

    key = random_key
    DETECTOR_R = detector_bounds['r']
    DETECTOR_H = detector_bounds['H']

    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=DETECTOR_R * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-DETECTOR_H/2 * fraction, maxval=DETECTOR_H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])

    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

    true_energy = photon_data['energy']
    TRUE_T0 = jax.random.uniform(key, shape=(), minval=-15.0, maxval=15.0)
    true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=float(TRUE_T0))

    original_direction = jnp.array([0.0, 0.0, 1.0])
    true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)
    rotation_axis = jnp.cross(original_direction, true_direction_norm)
    axis_norm = jnp.linalg.norm(rotation_axis)
    rotation_axis = jnp.where(axis_norm < 1e-6, jnp.array([1.0, 0.0, 0.0]), rotation_axis / (axis_norm + 1e-8))
    rotation_angle = jnp.arccos(jnp.clip(jnp.dot(original_direction, true_direction_norm), -1.0, 1.0))
    
    photon_data['rotation_axis'] = rotation_axis
    photon_data['rotation_angle'] = rotation_angle
    photon_data['apply_rotation'] = jnp.array(True)
    photon_data['apply_translation'] = jnp.array(True)
    photon_data['translation_vector'] = true_position
    
    key, _ = jax.random.split(key)
    true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))
    hit_counts, hit_times_raw = true_data
    hit_times = hit_times_raw + TRUE_T0

    return {
        'event_idx': event_idx, 'entry_idx': entry_idx, 'true_energy': float(true_energy),
        'true_position': np.array(true_position), 'true_direction': np.array(true_direction),
        'TRUE_T0': float(TRUE_T0), 'true_data': true_data, 'hit_times': hit_times,
        'hit_counts': hit_counts, 'photon_data': photon_data
    }

print('Event generation function defined.')

## Cell 5: Combined Loss Function

In [ ]:
TAU_TIME = 0.15  # Fixed tau for first_arrival_nll

def create_combined_loss_function(prediction_simulator, nrays_float, num_detectors, detector_points):
    '''Create combined loss function with gradient (likelihood-based).'''

    @jit
    def combined_product_loss(params, observed_times, observed_counts, key):
        position = params[:3]
        t0 = params[3]
        theta = params[4]
        phi = params[5]
        energy = params[6]

        # Simulate with t0=0
        track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))
        log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)

        # Charge loss (Poisson NLL)
        charge_loss = poisson_nll(observed_counts, total_charge)

        # Time loss (First-arrival NLL with shifted observations)
        t_obs_shifted = observed_times - t0
        time_nll = first_arrival_nll(
            log_w, flat_times, flat_indices,
            t_obs_shifted, TAU_TIME, num_detectors)

        hit_mask = observed_counts > 0
        n_hit = jnp.sum(hit_mask) + 1e-8
        time_loss = jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit

        # Vertex loss with dynamic tau_vtx
        tau_vtx = jax.lax.stop_gradient(
            TAU_VTX_PARAM_A * nrays_float + TAU_VTX_PARAM_B * energy + TAU_VTX_PARAM_C
        )
        tau_vtx = jnp.clip(tau_vtx, 0.05, 0.95)

        vertex_loss = origin_time_loss_configurable(
            jax.lax.stop_gradient(position), detector_points,
            observed_times, observed_counts, t0, tau=tau_vtx
        )

        # 3-term combined loss
        c, t, v, s = charge_loss, time_loss, vertex_loss, 0.
        combined = (jnp.sqrt((c + s) * (t + s) * (v + s)) +
                    jnp.sqrt((c + s) * jax.lax.stop_gradient((t + s) * (v + s))) +
                    jnp.sqrt((v + s) * jax.lax.stop_gradient((t + s) * (c + s))))

        return combined, (charge_loss, time_loss, vertex_loss)

    combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))
    return combined_grad_fn, combined_product_loss

# Create the loss function
combined_grad_fn, combined_product_loss = create_combined_loss_function(
    prediction_simulator, float(Nphot), NUM_DETECTORS, detector_points
)
print('Combined loss function created (likelihood-based).')

## Cell 6: Smeared Initial Guess & Adam Optimization

In [ ]:
def generate_smeared_initial_params(true_position, true_direction, true_energy, TRUE_T0,
                                     position_sigma=5.0,      # meters
                                     direction_sigma=15.0,    # degrees
                                     energy_sigma_frac=0.3,   # fraction (0.3 = 30%)
                                     t0_sigma=0.0,            # ns
                                     seed=None):
    """
    Generate initial parameters by smearing around true values.
    
    Parameters:
    -----------
    true_position : array
        True vertex position [x, y, z] in meters
    true_direction : array
        True direction unit vector [dx, dy, dz]
    true_energy : float
        True energy in MeV
    TRUE_T0 : float
        True t0 in ns
    position_sigma : float
        Standard deviation for position smearing in meters
    direction_sigma : float
        Standard deviation for direction smearing in degrees
    energy_sigma_frac : float
        Fractional standard deviation for energy (0.3 = 30%)
    t0_sigma : float
        Standard deviation for t0 smearing in ns
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    jnp.array
        Initial parameters [x, y, z, t0, theta, phi, energy]
    dict
        Smearing info with initial errors
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Smear position (Gaussian)
    init_position = true_position + np.random.normal(0, position_sigma, 3)
    
    # Clip position to detector bounds
    init_position[0] = np.clip(init_position[0], -DETECTOR_R * 0.9, DETECTOR_R * 0.9)
    init_position[1] = np.clip(init_position[1], -DETECTOR_R * 0.9, DETECTOR_R * 0.9)
    init_position[2] = np.clip(init_position[2], -DETECTOR_H/2 * 0.9, DETECTOR_H/2 * 0.9)
    
    # Smear direction (rotate by random angle around random axis)
    direction_sigma_rad = np.radians(direction_sigma)
    
    # Generate random rotation: sample angle from Rayleigh distribution
    # (gives roughly Gaussian-like angular deviation)
    rotation_angle = np.abs(np.random.normal(0, direction_sigma_rad))
    
    # Random rotation axis perpendicular to true direction
    random_vec = np.random.randn(3)
    random_vec = random_vec / np.linalg.norm(random_vec)
    perp_axis = np.cross(true_direction, random_vec)
    if np.linalg.norm(perp_axis) < 1e-6:
        random_vec = np.array([1, 0, 0]) if abs(true_direction[0]) < 0.9 else np.array([0, 1, 0])
        perp_axis = np.cross(true_direction, random_vec)
    perp_axis = perp_axis / np.linalg.norm(perp_axis)
    
    # Rodrigues rotation formula
    cos_a = np.cos(rotation_angle)
    sin_a = np.sin(rotation_angle)
    init_direction = (true_direction * cos_a + 
                      np.cross(perp_axis, true_direction) * sin_a +
                      perp_axis * np.dot(perp_axis, true_direction) * (1 - cos_a))
    init_direction = init_direction / np.linalg.norm(init_direction)
    
    # Convert to spherical angles
    init_theta, init_phi = cartesian_to_spherical(init_direction)
    
    # Smear energy (relative Gaussian)
    energy_sigma = true_energy * energy_sigma_frac
    init_energy = true_energy + np.random.normal(0, energy_sigma)
    init_energy = np.clip(init_energy, 300.0, 2000.0)
    
    # Smear t0 (Gaussian)
    init_t0 = TRUE_T0 + np.random.normal(0, t0_sigma)
    init_t0 = np.clip(init_t0, -20.0, 20.0)
    
    # Calculate initial errors
    init_position_error = np.linalg.norm(init_position - true_position)
    cos_angle = np.clip(np.dot(init_direction, true_direction), -1.0, 1.0)
    init_direction_error = np.degrees(np.arccos(cos_angle))
    init_energy_error = abs(init_energy - true_energy)
    init_t0_error = abs(init_t0 - TRUE_T0)
    
    initial_params = jnp.array([
        init_position[0], init_position[1], init_position[2],
        init_t0, init_theta, init_phi, init_energy
    ])
    
    smearing_info = {
        'position_sigma': position_sigma,
        'direction_sigma': direction_sigma,
        'energy_sigma_frac': energy_sigma_frac,
        't0_sigma': t0_sigma,
        'init_position_error': init_position_error,
        'init_direction_error': init_direction_error,
        'init_energy_error': init_energy_error,
        'init_t0_error': init_t0_error,
    }
    
    return initial_params, smearing_info


def run_adam_optimization(initial_params,
                          observed_times, observed_counts,
                          true_energy, true_position, true_direction, TRUE_T0,
                          max_iterations=500, verbosity=1):
    """Adam optimizer with likelihood-based loss."""
    if verbosity >= 1:
        print("=" * 60)
        print("ADAM OPTIMIZATION (likelihood-based)")
        print("=" * 60)
    
    true_theta, true_phi = cartesian_to_spherical(true_direction)
    
    ADAM_LEARNING_RATE = config['adam_optimizer']['learning_rate']
    ADAM_B1 = config['adam_optimizer']['b1']
    ADAM_B2 = config['adam_optimizer']['b2']
    ADAM_EPS = config['adam_optimizer']['eps']
    damping_factor = config['optimization_params']['damping_factor']
    tolerance = 1e-6
    
    POS_LR_SCALE = config['learning_rates']['position_learning_rate']
    DIR_LR_SCALE = config['learning_rates']['direction_learning_rate']*5.
    T0_LR_SCALE = config['learning_rates']['t0_learning_rate']
    ENE_LR_SCALE = config['learning_rates']['energy_learning_rate']
    
    update_scales = jnp.array([POS_LR_SCALE, POS_LR_SCALE, POS_LR_SCALE, T0_LR_SCALE, DIR_LR_SCALE, DIR_LR_SCALE, ENE_LR_SCALE])
    
    optimizer = optax.adam(learning_rate=ADAM_LEARNING_RATE, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    current_params = initial_params.copy()
    
    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [], 'charge_losses': [], 'time_losses': [],
        'vertex_losses': [], 'position_errors': [], 'direction_errors': [],
        't0_errors': [], 'energy_errors': [],
    }
    
    opt_key = jax.random.PRNGKey(12345)
    current_damping_w = 5.0
    adam_start_time = time.time()
    grad_norm = 1.0

    for iteration in range(max_iterations):
        opt_key, _ = jax.random.split(opt_key)
        
        (combined_loss, (charge_loss_val, time_loss_val, vertex_loss_val)), grad = combined_grad_fn(
            current_params, observed_times, observed_counts, opt_key
        )
        
        if jnp.any(jnp.isnan(grad)):
            grad = jnp.nan_to_num(grad, nan=0.0)
        
        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break
        
        updates, opt_state = optimizer.update(grad, opt_state, current_params)
        current_damping_w *= damping_factor
        scaled_updates = updates * update_scales * current_damping_w
        
        current_params = optax.apply_updates(current_params, scaled_updates)
        
        current_params = jnp.array([
            jnp.clip(current_params[0], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[1], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
            jnp.clip(current_params[2], -DETECTOR_H/2 * 0.95, DETECTOR_H/2 * 0.95),
            jnp.clip(current_params[3], -20.0, 20.0),
            current_params[4], current_params[5],
            jnp.clip(current_params[6], 300.0, 2000.0)
        ])
        
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_direction = spherical_to_cartesian(current_params[4], current_params[5])
        current_energy = current_params[6]
        
        position_error = float(jnp.linalg.norm(current_position - true_position))
        t0_error = float(abs(current_t0 - TRUE_T0))
        energy_error = float(abs(current_energy - true_energy))
        cos_angle = np.clip(np.dot(np.array(current_direction), np.array(true_direction)), -1.0, 1.0)
        direction_error = float(np.degrees(np.arccos(cos_angle)))
        
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['charge_losses'].append(float(charge_loss_val))
        history['time_losses'].append(float(time_loss_val))
        history['vertex_losses'].append(float(vertex_loss_val))
        history['position_errors'].append(position_error)
        history['direction_errors'].append(direction_error)
        history['t0_errors'].append(t0_error)
        history['energy_errors'].append(energy_error)
        
        if verbosity >= 1 and ((iteration + 1) % 100 == 0 or iteration == 0):
            print(f"  Iter {iteration}: loss={combined_loss:.4f}, pos_err={position_error:.2f}m, dir_err={direction_error:.1f}deg, E_err={energy_error:.0f}MeV")
    
    adam_end_time = time.time()
    
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)
    
    final_position_error = float(jnp.linalg.norm(final_position - true_position))
    final_t0_error = float(abs(final_t0 - TRUE_T0))
    final_energy_error = float(abs(final_energy - true_energy))
    cos_angle = np.clip(np.dot(np.array(final_direction), np.array(true_direction)), -1.0, 1.0)
    final_direction_error = float(np.degrees(np.arccos(cos_angle)))
    
    if verbosity >= 1:
        print(f"\n  Optimization completed in {adam_end_time - adam_start_time:.2f}s ({iteration+1} iterations)")
        print(f"  Final position error: {final_position_error:.3f} m")
        print(f"  Final direction error: {final_direction_error:.2f} deg")
        print(f"  Final energy error: {final_energy_error:.1f} MeV")
    
    return {
        'initial_params': np.array(initial_params), 'final_params': np.array(current_params),
        'final_position': np.array(final_position), 'final_direction': np.array(final_direction),
        'final_theta': float(final_theta), 'final_phi': float(final_phi),
        'final_t0': float(final_t0), 'final_energy': float(final_energy),
        'final_position_error': final_position_error, 'final_direction_error': final_direction_error,
        'final_t0_error': final_t0_error, 'final_energy_error': final_energy_error,
        'final_combined_loss': history['combined_losses'][-1] if history['combined_losses'] else float('inf'),
        'final_charge_loss': history['charge_losses'][-1] if history['charge_losses'] else float('inf'),
        'final_time_loss': history['time_losses'][-1] if history['time_losses'] else float('inf'),
        'adam_optimization_time': adam_end_time - adam_start_time,
        'total_iterations': len(history['parameters']) - 1,
        'converged': grad_norm < tolerance,
        'history': history
    }

print('Smeared initialization and Adam optimization functions defined (likelihood-based).')

## Cell 7: Run Optimization from Smeared Initial Guess

In [ ]:
# =====================================================================
# SMEARING CONFIGURATION - Adjust these to control initial distance from truth
# =====================================================================
POSITION_SIGMA = 5.0       # meters - position smearing
DIRECTION_SIGMA = 155.0     # degrees - direction smearing  
ENERGY_SIGMA_FRAC = 0.4    # fraction - energy smearing (0.4 = 40%)
T0_SIGMA = 0.0             # ns - t0 smearing
MAX_ITERATIONS = 500       # more iterations for distant start

# =====================================================================
# Main Processing
# =====================================================================
all_event_results = []
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

print(f'Processing {N_EVENTS} event(s) with smeared initial guess...')
print(f'Smearing: pos={POSITION_SIGMA}m, dir={DIRECTION_SIGMA}deg, E={ENERGY_SIGMA_FRAC*100:.0f}%, t0={T0_SIGMA}ns')
print('=' * 80)

for event_idx in range(N_EVENTS):
    event_start_time = time.time()
    print(f'\n{"="*80}')
    print(f'EVENT {event_idx}')
    print(f'{"="*80}')
    
    try:
        event_key = event_keys[event_idx]
        max_attempts = 10
        endpoint_valid = False
        
        for attempt in range(max_attempts):
            event_data = generate_event_data(
                event_idx, event_key, data_dir,
                data_simulator, detector_bounds, fraction=0.9
            )
            endpoint_valid = check_track_endpoint_in_detector(
                event_data['true_position'], event_data['true_direction'],
                event_data['true_energy'], range_params, detector_bounds, fraction=0.9
            )
            if endpoint_valid:
                break
            event_key, _ = jax.random.split(event_key)
        
        if not endpoint_valid:
            print(f'  ERROR: Could not generate valid event')
            continue
        
        true_position = event_data['true_position']
        true_direction = event_data['true_direction']
        true_energy = event_data['true_energy']
        TRUE_T0 = event_data['TRUE_T0']
        true_data = event_data['true_data']
        
        # Get observed data (full arrays, not masked - the loss function handles masking internally)
        observed_times = event_data['hit_times']
        observed_counts = event_data['hit_counts']
        
        hit_mask = observed_counts > 0
        n_hits = int(jnp.sum(hit_mask))
        
        print(f'  True position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}]')
        print(f'  True direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]')
        print(f'  True energy: {true_energy:.1f} MeV')
        print(f'  True t0: {TRUE_T0:.2f} ns')
        print(f'  N hits: {n_hits}')
        
        # Generate smeared initial parameters (SKIP GRID SEARCH)
        print(f'\n  Generating smeared initial guess...')
        initial_params, smearing_info = generate_smeared_initial_params(
            true_position, true_direction, true_energy, TRUE_T0,
            position_sigma=POSITION_SIGMA,
            direction_sigma=DIRECTION_SIGMA,
            energy_sigma_frac=ENERGY_SIGMA_FRAC,
            t0_sigma=T0_SIGMA,
            seed=event_idx + 1234
        )
        
        print(f'  Initial errors:')
        print(f'    Position: {smearing_info["init_position_error"]:.2f} m')
        print(f'    Direction: {smearing_info["init_direction_error"]:.1f} deg')
        print(f'    Energy: {smearing_info["init_energy_error"]:.1f} MeV')
        print(f'    t0: {smearing_info["init_t0_error"]:.2f} ns')
        
        # Run Adam optimization from smeared start (likelihood-based)
        opt_results = run_adam_optimization(
            initial_params,
            observed_times,
            observed_counts,
            true_energy,
            true_position, true_direction, TRUE_T0,
            max_iterations=MAX_ITERATIONS, verbosity=1
        )
        
        event_end_time = time.time()
        
        event_result = {
            'event_data': {
                'event_idx': event_data['event_idx'], 'true_energy': event_data['true_energy'],
                'true_position': event_data['true_position'], 'true_direction': event_data['true_direction'],
                'TRUE_T0': event_data['TRUE_T0'], 'true_data': event_data['true_data'],
                'observed_times': observed_times, 'observed_counts': observed_counts,
            },
            'smearing_info': smearing_info,
            'stage4': opt_results,  # Keep same key for compatibility with GIF generation
            'total_event_time': event_end_time - event_start_time
        }
        all_event_results.append(event_result)
        
        print(f'\n  Event {event_idx} completed in {event_end_time - event_start_time:.2f}s')
        print(f'  Improvement: pos {smearing_info["init_position_error"]:.2f}m -> {opt_results["final_position_error"]:.3f}m')
        print(f'               dir {smearing_info["init_direction_error"]:.1f}deg -> {opt_results["final_direction_error"]:.2f}deg')
        
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()
        continue

print(f'\n{"="*80}')
print(f'Completed {len(all_event_results)} event(s)')

## Cell 8: GIF Generation Functions

Functions to create optimization video frames and combine them into an animated GIF.

In [ ]:
def process_simulator_output_for_visualization(sim_output, num_detectors):
    """
    Convert new simulator output (log_w, flat_times, flat_indices, total_charge)
    to visualization format (charges, times).
    
    For times, computes weighted mean arrival time per detector.
    """
    log_w, flat_times, flat_indices, total_charge = sim_output
    
    # Charges are directly available
    sim_charges = np.array(total_charge)
    
    # Compute weighted mean time per detector
    weights = np.exp(np.array(log_w))
    flat_times_np = np.array(flat_times)
    flat_indices_np = np.array(flat_indices).astype(int)
    
    # Initialize arrays for weighted sum and weight sum
    weighted_time_sum = np.zeros(num_detectors)
    weight_sum = np.zeros(num_detectors)
    
    # Accumulate weighted times per detector
    np.add.at(weighted_time_sum, flat_indices_np, weights * flat_times_np)
    np.add.at(weight_sum, flat_indices_np, weights)
    
    # Compute weighted mean (avoid division by zero)
    sim_times = np.where(weight_sum > 0, weighted_time_sum / weight_sum, 0.0)
    
    return sim_charges, sim_times


def create_optimization_frames(event_result, prediction_simulator, display_function,
                               iteration_step=5, max_iteration=None, use_time=False,
                               fixed_colorbar=True, colorbar_margin=30, output_folder='optimization_frames'):
    '''
    Create frames showing optimization progression.
    For time visualization, channels with zero charge are masked (delta time = 0).
    '''
    event_idx = event_result['event_data']['event_idx']
    true_data = event_result['event_data']['true_data']
    param_history = event_result['stage4']['history']['parameters']

    frame_type = 'time' if use_time else 'charge'
    folder_path = os.path.join(output_folder, f'event_{event_idx}', frame_type)
    os.makedirs(folder_path, exist_ok=True)

    total_iterations = len(param_history)
    end_iteration = min(800, total_iterations) if max_iteration is None else min(max_iteration, total_iterations)

    print(f'Creating {frame_type} frames for event {event_idx}...')
    print(f'Total iterations: {total_iterations}, processing: 0 to {end_iteration}')
    print(f'Saving every {iteration_step} iterations')

    sim_key = jax.random.PRNGKey(42)

    # Get true data
    true_charges, true_times = true_data
    true_charges = np.array(true_charges)
    true_times = np.array(true_times)

    colorbar_range = None
    if fixed_colorbar:
        print('Calculating colorbar range from first frame...')
        first_params = param_history[0]
        position = first_params[:3]
        theta = first_params[4]
        phi = first_params[5]
        energy = first_params[6]
        track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))

        first_sim_output = prediction_simulator(track, sim_key)
        sim_charges, sim_times = process_simulator_output_for_visualization(first_sim_output, NUM_DETECTORS)

        if use_time:
            # Mask: only include channels where BOTH true and sim have non-zero charge
            valid_mask = (true_charges > 0) & (sim_charges > 0)
            differences = np.zeros_like(sim_times)
            differences[valid_mask] = sim_times[valid_mask] - true_times[valid_mask]
        else:
            differences = sim_charges - true_charges

        # Filter out any extreme values
        valid_diffs = differences[np.abs(differences) < 1e5]
        if len(valid_diffs) > 0:
            max_abs_value = np.max(np.abs(valid_diffs))
        else:
            max_abs_value = 1.0

        max_abs_value = 20    
        colorbar_range = max_abs_value * (1 + colorbar_margin / 100)
        print(f'  Colorbar range: +/- {colorbar_range:.2e}')

    frame_count = 0

    for iteration in range(0, end_iteration, iteration_step):
        params = param_history[iteration]
        position = params[:3]
        theta = params[4]
        phi = params[5]
        energy = params[6]
        track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))

        sim_output = prediction_simulator(track, sim_key)
        sim_charges, sim_times = process_simulator_output_for_visualization(sim_output, NUM_DETECTORS)
        sim_data = (sim_charges, sim_times)

        if use_time:
            # Apply masking for time visualization
            # Mask: only show time diff where BOTH have non-zero charge
            valid_mask = (true_charges > 0) & (sim_charges > 0)

            # Create masked times: set to same value where invalid (so diff = 0)
            masked_true_times = np.where(valid_mask, true_times, 0.0)
            masked_sim_times = np.where(valid_mask, sim_times, 0.0)

            # Pass masked data to display function
            masked_true_data = (true_charges, masked_true_times)
            masked_sim_data = (sim_charges, masked_sim_times)

            filename = os.path.join(folder_path, f'frame_{frame_count:04d}_iter_{iteration:04d}.png')

            display_function(
                true_data=masked_true_data, sim_data=masked_sim_data, file_name=filename,
                plot_time=use_time, align_time=False, colorbar_range=colorbar_range
            )
        else:
            filename = os.path.join(folder_path, f'frame_{frame_count:04d}_iter_{iteration:04d}.png')

            display_function(
                true_data=true_data, sim_data=sim_data, file_name=filename,
                plot_time=use_time, align_time=use_time, colorbar_range=colorbar_range
            )

        plt.close('all')
        frame_count += 1

        if iteration % (iteration_step * 10) == 0:
            print(f'  Generated frame {frame_count} (iteration {iteration})')

    print(f'Completed! Generated {frame_count} frames in {folder_path}')
    return folder_path


def create_gif(frames_folder, output_path, total_duration=10.0, loop=0, optimize=True):
    '''Create animated GIF from frames.'''
    frame_files = sorted(glob.glob(os.path.join(frames_folder, 'frame_*.png')))

    if len(frame_files) == 0:
        raise ValueError(f'No frame files found in {frames_folder}')

    print(f'Found {len(frame_files)} frames')

    frame_duration = int((total_duration / len(frame_files)) * 1000)
    print(f'Frame duration: {frame_duration}ms each')

    frames = [Image.open(f) for f in frame_files]

    output_file = f'{output_path}.gif'
    frames[0].save(
        output_file, save_all=True, append_images=frames[1:],
        duration=frame_duration, loop=loop, optimize=optimize
    )

    print(f'GIF created: {output_file}')
    return output_file


def clean_optimization_frames(base_folder='optimization_frames', event_idx=None):
    '''Clean optimization frames.'''
    if not os.path.exists(base_folder):
        print(f'Base folder {base_folder} does not exist.')
        return

    if event_idx is not None:
        event_folder = os.path.join(base_folder, f'event_{event_idx}')
        if os.path.exists(event_folder):
            shutil.rmtree(event_folder)
            print(f'Deleted: {event_folder}')
    else:
        shutil.rmtree(base_folder)
        print('Deleted entire optimization_frames folder')


print('GIF generation functions defined.')

## Cell 9: Generate Optimization GIF

In [ ]:
if len(all_event_results) > 0:
    EVENT_TO_VISUALIZE = 0
    event_result = all_event_results[EVENT_TO_VISUALIZE]
    
    display_function = create_detector_comparison_display(
        json_filename=default_json_filename, sparse=False
    )
    
    # Use Agg backend to prevent display
    matplotlib.use('Agg')
    
    clean_optimization_frames(event_idx=EVENT_TO_VISUALIZE)
    
    print('\n' + '='*60)
    print('GENERATING OPTIMIZATION FRAMES')
    print('='*60)
    
    folder_path = create_optimization_frames(
        event_result, prediction_simulator, display_function,
        iteration_step=10, max_iteration=None, use_time=False,
        fixed_colorbar=True, colorbar_margin=30
    )
    
    print('\n' + '='*60)
    print('CREATING ANIMATED GIF')
    print('='*60)
    
    figures_dir = Path('figures')
    figures_dir.mkdir(parents=True, exist_ok=True)
    
    gif_file = create_gif(
        frames_folder=folder_path,
        output_path=f'figures/optimization_event_{EVENT_TO_VISUALIZE}_charge',
        total_duration=4.8
    )
    
    print(f'\nGIF saved to: {gif_file}')
else:
    print('No event results available.')

## Cell 10: Display Results Summary

In [ ]:
# Switch back to inline display
%matplotlib inline

if len(all_event_results) > 0:
    event = all_event_results[0]
    history = event['stage4']['history']
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    axes[0, 0].semilogy(history['combined_losses'])
    axes[0, 0].set_title('Combined Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(history['position_errors'])
    axes[0, 1].set_title('Position Error (m)')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(history['direction_errors'])
    axes[1, 0].set_title('Direction Error (deg)')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(history['energy_errors'])
    axes[1, 1].set_title('Energy Error (MeV)')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figures/optimization_convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nFinal Results:')
    print(f'  Position error: {event["stage4"]["final_position_error"]:.3f} m')
    print(f'  Direction error: {event["stage4"]["final_direction_error"]:.2f} deg')
    print(f'  Energy error: {event["stage4"]["final_energy_error"]:.1f} MeV')